# Phase 2 walkthrough — feature-pair refactor

End-to-end check of the new A→B→C path being built alongside the existing rounded kernel.

Status today (the notebook stays in sync as installments land):
- **Phase A** — `emitFeaturePairsKernel`: emits feature-pair candidates from the per-vertex ball list, canonical by polygon shape. ✓ done.
- **Phase B installment 1** — `detectCrossingsKernel`: edge-edge dispatch only, on backbone edges (not flat-segment edges yet). ✓ done.
- **Phase B installment 2** — arc-edge / arc-arc dispatch + flat-segment edges. *In progress.*
- **Phase C** — sort crossings + per-feature walk. *Not started.*

The notebook mirrors the early cells of `biArcTesting.ipynb` to build a rounded packing, then drops down into the new A/B entry points to sanity-check the plumbing.

In [ ]:
import numpy as np
import sys, os
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath('../pyCudaPolygon'))
import pyCudaPolygon as pcp

## 1. Build a softBody packing and relax

Same recipe as `biArcTesting.ipynb`: 64 polygons of 32 vertices, equal perimeter (bidisperse, ratio 1.4), small δ. The relaxation gives realistic positions that the rounded model will then probe for overlaps.

In [ ]:
N, n = 64, 32
seed = 42
kappa = 3.7
rho = 0.001

m = pcp.model(size=N*n, seed=seed)
m.setModelEnum('softBody')
m.generateRandomPolygons(N, n)
m.setBiPerimeters(kappa)
m.setDelta(rho)
m.setStiffness(1)
m.setCompressibility(1)
m.initializeNeighborBall()
m.updateForceEnergy()
print(f'initial softBody E = {m.getEnergy():.4e}   |F|max = {m.getMaxUnbalancedForce():.4e}')

In [ ]:
# Short softBody relaxation so polygons sit at sensible shapes before we add overlap.
m.minimizeFIRE(maxForceThreshold=1e-12, maxSteps=2000, progressBar=False)
print(f'after softBody FIRE: E = {m.getEnergy():.4e}   |F|max = {m.getMaxUnbalancedForce():.4e}')

In [ ]:
# Compress to phi=1.1 so the rounded model actually finds overlaps.
m.setPhi(1.1)
m.updatePolygonGeometry()
viol = m.getConstraintViolation()
print(f'after setPhi(1.1):   rmsAreaViol = {viol["rmsAreaViolation"]:.3e}   '
      f'rmsEdgeViol = {viol["rmsEdgeViolation"]:.3e}')

## 2. Switch to the rounded model

The existing rounded kernel (`updateForceEnergyRoundedKernel`) is still the production path. We initialize the ball list (which Phase A reads) and run one force evaluation through the existing kernel so we have a reference.

In [ ]:
m.setModelEnum('rounded')
m.initializeNeighborBall()
m.updatePolygonGeometry()
m.updateNeighborBall()
m.updateForceEnergy()
print(f'rounded init:        E = {m.getEnergy():.4e}   |F|max = {m.getMaxUnbalancedForce():.4e}')
print(f'ball list:           mean count = {np.mean(m.getNumBallNeighbors()):.1f}   '
      f'rebuilds = {m.getBallRebuildCount()}')
print(f'searchFactor = {m.getSearchFactor():.2f}   maxEdgeLength = {m.getMaxEdgeLength():.4e}')

## 3. Phase A: emit feature-pair candidates

`runFeaturePairPhaseA(featureSetSize)` walks the ball list and emits `featureSetSize²` feature pairs per cross-polygon vertex pair (canonical: only when `shapeId[i] < shapeId[j]`, so each polygon-pair is emitted from one side). Output sizes scale as expected:
- `featureSetSize=1` → 1 pair per (i,j): EDGE × EDGE only (normal model footprint)
- `featureSetSize=2` → 4 pairs per (i,j): the 4 combos of {EDGE, ARC1} (single-arc rounded)
- `featureSetSize=3` → 9 pairs per (i,j): {EDGE, ARC1, ARC2} (biarc)

In [ ]:
for fs in (1, 2, 3):
    n_pairs = m.runFeaturePairPhaseA(fs)
    print(f'featureSetSize = {fs}:  emitted {n_pairs:6d} pairs   '
          f'(capacity = {m.getFeaturePairCapacity()})')

In [ ]:
# Sanity: featureSetSize=2 should emit 4x the count of featureSetSize=1.
n1 = m.runFeaturePairPhaseA(1)
n2 = m.runFeaturePairPhaseA(2)
n3 = m.runFeaturePairPhaseA(3)
print(f'n2 / n1 = {n2/n1:.3f}   (expected 4.0)')
print(f'n3 / n1 = {n3/n1:.3f}   (expected 9.0)')
assert n2 == 4 * n1
assert n3 == 9 * n1
print('OK: Phase A scaling is featureSetSize^2.')

## 4. Phase B: detect crossings (edge-edge only, installment 1)

`runFeaturePairPhaseB(delta)` consumes Phase A's pairs, dispatches per type to the intersection routines in `intersect.cuh`, and emits a `Crossing` record per detected intersection. Currently only the `(EDGE, EDGE)` combo is dispatched (the others are silently dropped pending installment 2).

In [ ]:
# Reference: the existing intersection finder (updateNeighbors) walks the
# cell stencil. We didn't initialize cells when we set up the rounded model
# (the new flow uses balls only) so we have to do it now for the comparison.
m.initializeNeighborCells()
m.updateNeighborCells()
m.updateNeighbors()
existing_total = int(np.asarray(m.getNumNeighbors(), dtype=np.int64).sum())
print(f'existing updateNeighbors total crossings:  {existing_total}')

In [ ]:
# New path: Phase A (featureSetSize=1) + Phase B (edge-edge). Pass the same
# delta as the existing kernel so the endpoint-exclusion bounds match.
m.runFeaturePairPhaseA(1)
delta = m.getDelta()
phaseB_count = m.runFeaturePairPhaseB(delta)
print(f'Phase A+B (fs=1) total crossings:          {phaseB_count}')
print(f'absolute difference:                       {abs(phaseB_count - existing_total)}')
assert abs(phaseB_count - existing_total) <= 1, \
    f'Phase A+B count {phaseB_count} differs from existing {existing_total}'
print('OK: edge-edge count matches existing intersection finder.')

In [ ]:
# featureSetSize=2: arc combos still silently dropped. We expect the same
# crossing count as fs=1 (because only EDGE x EDGE is dispatched today).
m.runFeaturePairPhaseA(2)
phaseB_count_fs2 = m.runFeaturePairPhaseB(delta)
print(f'Phase A+B (fs=2, edge-only dispatch):      {phaseB_count_fs2}')
assert phaseB_count_fs2 == phaseB_count, (
    f'fs=2 count {phaseB_count_fs2} should equal fs=1 count {phaseB_count} '
    'until arc dispatch lands in installment 2.'
)
print('OK: fs=2 currently matches fs=1 (arc dispatch pending).')

## 5. Round-trip the candidate buffer back to balls

Section 4 deliberately repopulated the unified candidate buffer from the cell stencil so we could compare against the legacy `updateNeighbors`. The rounded kernel reads that buffer, so its energy momentarily reflects the cell-built reach. Re-run the ball builder to restore the original (ball-built) candidate set and confirm the rounded energy returns to its section 2 value.

In [ ]:
# Section 4 called initializeNeighborCells() so we could compare against the
# legacy updateNeighbors. That re-fills the unified candidate buffer using
# the cell stencil (which has a different reach than the ball builder), so
# the rounded kernel's energy temporarily changes. Re-build the ball list
# and confirm the kernel returns to the same value it had at the end of
# section 2.
E_before = 16.382        # from section 2's print
m.updateNeighborBall()   # repopulates the candidate buffer from the ball builder
m.updateForceEnergy()
E_after = m.getEnergy()
print(f'E at section 2:                       {E_before:.4f}')
print(f'E after restoring ball-built buffer:  {E_after:.4f}')
print(f'absolute diff:                        {abs(E_after - E_before):.3e}')
assert abs(E_after - E_before) < 1e-2, 'existing rounded path failed to round-trip'
print('OK: existing rounded path round-trips after restoring the ball-built candidate buffer.')

## 6. What's next

When installment 2 of Phase B lands, the assertion in section 4's last cell will need to change — `featureSetSize=2` will then produce *more* crossings than `featureSetSize=1` (because arc-edge and arc-arc dispatch starts contributing).

When Phase C lands, the new path will be runnable end-to-end and FD-validatable against the existing monolith.